## mount drive

In [1]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')
print("✅ Google Drive connected successfully!")
print("Your files are now accessible at: /content/drive/MyDrive/")

Mounted at /content/drive
✅ Google Drive connected successfully!
Your files are now accessible at: /content/drive/MyDrive/


## imports and packages

In [3]:
!pip install faiss-cpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 69.2 MB/s eta 0:00:00


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler
from transformers import (
    CLIPVisionModel, CLIPTextModel, CLIPTokenizer, CLIPProcessor,
    AutoModelForCausalLM, AutoTokenizer
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from PIL import Image
from sklearn.model_selection import train_test_split
import json
from tqdm import tqdm
import pickle
from typing import List, Dict, Tuple, Optional
import logging
from dataclasses import dataclass
import warnings
import faiss
import re
from google import genai
from google.genai import types
from google.colab import userdata
import requests
import time

In [ ]:
# Device configuration
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


## previous phases files

In [ ]:
drive_path = "/content/drive/MyDrive/"
phase_1_path = f"{drive_path}/NLP_HW3_phase1"
phase_2_path = f"{drive_path}/NLP_HW3_phase2"
phase_3_path = f"{drive_path}/NLP_HW3_phase3"

### phase 1

In [ ]:
!ls {phase_1_path}

drivers  HW_3.ipynb  mashahir  people_imgs


In [ ]:
!ls {phase_1_path}/mashahir

inputs	outputs


In [ ]:
!ls {phase_1_path}/mashahir/inputs

1.json	2.json	3.json	4.json	5.json	6.json	7.json	8.json


In [ ]:
!ls {phase_1_path}/mashahir/outputs

json_1	json_2	json_3	json_4	json_5	json_6	json_7	json_8


In [ ]:
!ls {phase_1_path}/mashahir/outputs/json_1/phase_2

0000_all_img_link_added.json  downloaded_images		image_retrieval.log
0000_all.json		      downloaded_images_filter


### phase 2

In [ ]:
!ls {phase_2_path}

FineTuned_LoRA_CLIP  Persian_Vector_DB


In [ ]:
!ls {phase_2_path}/"Persian_Vector_DB"

index.faiss  metadata.pkl


In [ ]:
!ls {phase_2_path}/"FineTuned_LoRA_CLIP"

additional_params.pt   checkpoint_epoch_4.pt  tokenizer
best_checkpoint.pt     config.json	      training_stats.json
checkpoint_epoch_1.pt  processor	      vision_encoder.pt
checkpoint_epoch_2.pt  projection_layers.pt
checkpoint_epoch_3.pt  text_lora


### phase 3 questions

In [ ]:
!ls {phase_3_path}

Image  Question_only_Text.json	questions_codes.zip  Question_Text_Image.json


## phase 3 inference implementations

### setting up environment

In [14]:
os.environ["GEMINI_API_KEY"] = userdata.get('Gemeni_API_KEY')

### helpers classes

In [15]:
class CLIPWithLoRAFixed(nn.Module):
    """CLIP wrapper with proper dimension handling"""

    def __init__(self, vision_model, text_model):
        super().__init__()
        self.vision_model = vision_model
        self.text_model = text_model

        self.vision_dim = vision_model.config.hidden_size if hasattr(vision_model.config, 'hidden_size') else 768
        self.text_dim = text_model.config.hidden_size if hasattr(text_model.config, 'hidden_size') else 512

        common_dim = 512
        self.vision_projection = nn.Linear(self.vision_dim, common_dim)
        self.text_projection = nn.Linear(self.text_dim, common_dim)

        device = next(vision_model.parameters()).device
        self.vision_projection = self.vision_projection.to(device)
        self.text_projection = self.text_projection.to(device)

        print(f"Initialized CLIPWithLoRAFixed: vision_dim={self.vision_dim}, text_dim={self.text_dim}, common_dim={common_dim}")

    def encode_image(self, pixel_values):
        """Encode images with proper dimension handling"""
        vision_output = self.vision_model(pixel_values=pixel_values)
        image_embeds = vision_output.pooler_output
        image_embeds = self.vision_projection(image_embeds)
        return vision_output, image_embeds

    def encode_text(self, input_ids, attention_mask):
        """Encode text with proper dimension handling"""
        if hasattr(self.text_model, 'base_model'):
            actual_text_model = self.text_model.base_model
            text_output = actual_text_model(input_ids=input_ids, attention_mask=attention_mask)
        else:
            text_output = self.text_model(input_ids=input_ids, attention_mask=attention_mask)

        text_embeds = text_output.pooler_output
        text_embeds = self.text_projection(text_embeds)
        return text_output, text_embeds


In [16]:
class LoRACLIPFineTuner:
    """FIXED: LoRA CLIP Fine-tuner with corrected save/load logic"""

    def __init__(self, model_name: str = "openai/clip-vit-base-patch32", device: str = None, use_mixed_precision: bool = False):
        self.model_name = model_name
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.use_mixed_precision = False  # Force disable
        self.scaler = None

        print(f"Loading CLIP model: {model_name}")
        self._load_models()

        self.temperature = nn.Parameter(torch.tensor(0.07, device=self.device, requires_grad=True))
        self.clip_model = None
        self.lora_applied_to_text_only = True  # TRACK WHICH MODEL HAS LORA

        self.training_stats = {
            'train_losses': [],
            'test_losses': [],
            'learning_rates': [],
            'temperatures': []
        }

        print(f"Loaded CLIP models on {self.device}")

    def _load_models(self):
        """Load CLIP models with error handling"""
        try:
            self.vision_encoder = CLIPVisionModel.from_pretrained(self.model_name).to(self.device)
            # self.vision_encoder = CLIPVisionModel.from_pretrained(self.model_name, use_fast=True).to(self.device)
            self.text_encoder = CLIPTextModel.from_pretrained(self.model_name).to(self.device)
            self.tokenizer = CLIPTokenizer.from_pretrained(self.model_name)
            self.processor = CLIPProcessor.from_pretrained(self.model_name)
            # self.processor = CLIPProcessor.from_pretrained(self.model_name, use_fast=True)
        except Exception as e:
            print(f"Failed to load CLIP models: {e}")
            raise

    def apply_lora(self, config):
        """Apply LoRA only to text encoder - FIXED VERSION"""
        print("Applying LoRA to text encoder only...")

        text_lora_config = LoraConfig(
            r=config.lora_r,
            lora_alpha=config.lora_alpha,
            target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
            lora_dropout=config.lora_dropout,
            bias="none",
            task_type=TaskType.FEATURE_EXTRACTION,
        )

        try:
            # Only apply LoRA to text encoder
            self.text_encoder = get_peft_model(self.text_encoder, text_lora_config)
            self.lora_applied_to_text_only = True  # Mark that only text has LoRA

            # Create wrapper model
            self.clip_model = CLIPWithLoRAFixed(
                self.vision_encoder,
                self.text_encoder
            )

            print("LoRA applied successfully to text encoder only")
            self.print_trainable_parameters()

        except Exception as e:
            print(f"Failed to apply LoRA: {e}")
            raise

    def print_trainable_parameters(self):
        """Print trainable parameters info"""
        vision_trainable = sum(p.numel() for p in self.vision_encoder.parameters() if p.requires_grad)
        text_trainable = sum(p.numel() for p in self.text_encoder.parameters() if p.requires_grad)
        vision_total = sum(p.numel() for p in self.vision_encoder.parameters())
        text_total = sum(p.numel() for p in self.text_encoder.parameters())

        print(f"Vision encoder: {vision_trainable:,}/{vision_total:,} trainable parameters ({100*vision_trainable/vision_total:.2f}%)")
        print(f"Text encoder: {text_trainable:,}/{text_total:,} trainable parameters ({100*text_trainable/text_total:.2f}%)")

    def contrastive_loss(self, image_embeddings, text_embeddings):
        """Enhanced contrastive loss"""
        image_embeddings = F.normalize(image_embeddings, dim=1)
        text_embeddings = F.normalize(text_embeddings, dim=1)

        temperature = torch.clamp(self.temperature, min=0.01, max=1.0)
        logits = torch.matmul(image_embeddings, text_embeddings.T) / temperature

        batch_size = logits.shape[0]
        labels = torch.arange(batch_size, device=self.device)

        loss_img = F.cross_entropy(logits, labels, label_smoothing=0.1)
        loss_text = F.cross_entropy(logits.T, labels, label_smoothing=0.1)

        return (loss_img + loss_text) / 2

    def train_epoch(self, dataloader, optimizer, scheduler, epoch, config):
        """FIXED training loop"""
        if self.clip_model is None:
            raise RuntimeError("LoRA not applied! Call apply_lora() first.")

        self.vision_encoder.train()
        self.text_encoder.train()
        total_loss = 0
        num_batches = 0

        optimizer.zero_grad()

        pbar = tqdm(dataloader, desc=f"Epoch {epoch}")
        for batch_idx, batch in enumerate(pbar):
            if batch is None:
                print(f"Skipping None batch at index {batch_idx}")
                continue

            try:
                input_ids = batch['input_ids'].to(self.device, non_blocking=True)
                attention_mask = batch['attention_mask'].to(self.device, non_blocking=True)
                pixel_values = batch['pixel_values'].to(self.device, non_blocking=True)

                text_output, text_embeds = self.clip_model.encode_text(input_ids, attention_mask)
                image_output, image_embeds = self.clip_model.encode_image(pixel_values)

                loss = self.contrastive_loss(image_embeds, text_embeds)
                loss = loss / config.accumulation_steps

                if torch.isnan(loss) or torch.isinf(loss):
                    print(f"Invalid loss at batch {batch_idx}: {loss.item()}")
                    continue

                loss.backward()

                if (batch_idx + 1) % config.accumulation_steps == 0:
                    total_norm = torch.nn.utils.clip_grad_norm_(
                        list(self.vision_encoder.parameters()) +
                        list(self.text_encoder.parameters()) +
                        [self.temperature],
                        max_norm=config.max_grad_norm
                    )

                    optimizer.step()
                    optimizer.zero_grad()
                    scheduler.step()

                total_loss += loss.item() * config.accumulation_steps
                num_batches += 1

                current_temp = torch.clamp(self.temperature, min=0.01, max=1.0).item()
                current_lr = scheduler.get_last_lr()[0] if hasattr(scheduler, 'get_last_lr') else config.learning_rate

                pbar.set_postfix({
                    'loss': f'{loss.item() * config.accumulation_steps:.3f}',
                    'temp': f'{current_temp:.3f}',
                    'lr': f'{current_lr:.2e}'
                })

                if batch_idx % 50 == 0:
                    torch.cuda.empty_cache()

            except RuntimeError as e:
                if "out of memory" in str(e):
                    print(f"CUDA OOM at batch {batch_idx}, skipping batch")
                    torch.cuda.empty_cache()
                    continue
                else:
                    print(f"Training error at batch {batch_idx}: {e}")
                    continue

        return total_loss / max(num_batches, 1)

    def test_epoch(self, dataloader):
        """FIXED test loop"""
        if dataloader is None:
            return float('inf')

        if self.clip_model is None:
            raise RuntimeError("LoRA not applied! Call apply_lora() first.")

        self.vision_encoder.eval()
        self.text_encoder.eval()
        total_loss = 0
        num_batches = 0

        with torch.no_grad():
            for batch in tqdm(dataloader, desc="Testing"):
                if batch is None:
                    continue

                try:
                    input_ids = batch['input_ids'].to(self.device, non_blocking=True)
                    attention_mask = batch['attention_mask'].to(self.device, non_blocking=True)
                    pixel_values = batch['pixel_values'].to(self.device, non_blocking=True)

                    text_output, text_embeds = self.clip_model.encode_text(input_ids, attention_mask)
                    image_output, image_embeds = self.clip_model.encode_image(pixel_values)

                    loss = self.contrastive_loss(image_embeds, text_embeds)

                    if not (torch.isnan(loss) or torch.isinf(loss)):
                        total_loss += loss.item()
                        num_batches += 1

                except RuntimeError as e:
                    if "out of memory" in str(e):
                        print("CUDA OOM during testing, skipping batch")
                        torch.cuda.empty_cache()
                        continue
                    else:
                        print(f"Test error: {e}")
                        continue

        return total_loss / max(num_batches, 1)

    def create_dataloaders(self, csv_path: str, image_folder: str, config):
        """Create fixed dataloaders"""
        train_samples, test_samples = self.create_data_splits(csv_path, image_folder)

        train_dataset = PersianCelebrityDataset(train_samples, self.tokenizer, self.processor)
        test_dataset = PersianCelebrityDataset(test_samples, self.tokenizer, self.processor)

        train_loader = create_fixed_dataloader(
            train_dataset,
            batch_size=config.batch_size,
            shuffle=True,
            num_workers=0
        )

        test_loader = None
        if len(test_dataset) > 0:
            test_loader = create_fixed_dataloader(
                test_dataset,
                batch_size=config.batch_size,
                shuffle=False,
                num_workers=0
            )

        return train_loader, test_loader

    def create_data_splits(self, csv_path: str, image_folder: str, test_split: float = 0.2):
        """Enhanced data splitting with better validation"""
        validation_result = DataValidator.validate_data_structure(csv_path, image_folder)

        if not validation_result['valid']:
            for error in validation_result['errors']:
                print(error)
            raise ValueError("Data validation failed")

        for warning in validation_result['warnings']:
            print(warning)

        stats = validation_result['stats']
        print(f"Data validation passed:")
        print(f"  - Total celebrities: {stats['total_celebrities']}")
        print(f"  - Available celebrities: {stats['available_celebrities']}")
        print(f"  - Total images: {stats['total_images']}")
        print(f"  - Avg images per celebrity: {stats['avg_images_per_celeb']:.1f}")

        data = pd.read_csv(csv_path)
        train_samples = []
        test_samples = []

        for _, row in data.iterrows():
            name = row['name']
            iteration_val = str(row["iteration"])
            celebrity_folder = os.path.join(image_folder, iteration_val.replace(".0", ""))
            print(celebrity_folder)
            if not os.path.exists(celebrity_folder):
                continue

            image_files = [f for f in os.listdir(celebrity_folder)
                          if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif', '.webp', '.tiff', '.tif'))]
            print(image_files)
            if len(image_files) < 1:
                continue
            elif len(image_files) == 1:
                train_samples.append({
                    'name': name,
                    'question': row['question'],
                    'answer': row['answer'],
                    'image_path': os.path.join(celebrity_folder, image_files[0])
                })
            else:
                image_files.sort()
                n_test = max(1, int(len(image_files) * test_split))
                test_images = image_files[-n_test:]
                train_images = image_files[:-n_test]

                for img_file in train_images:
                    train_samples.append({
                        'name': name,
                        'question': row['question'],
                        'answer': row['answer'],
                        'image_path': os.path.join(celebrity_folder, img_file)
                    })

                for img_file in test_images:
                    test_samples.append({
                        'name': name,
                        'question': row['question'],
                        'answer': row['answer'],
                        'image_path': os.path.join(celebrity_folder, img_file)
                    })

        print(f"Created {len(train_samples)} training samples and {len(test_samples)} test samples")

        if len(test_samples) == 0:
            print("No test samples created. Consider adding more images per celebrity.")

        return train_samples, test_samples

    def save_checkpoint(self, save_path: str, epoch: int, train_loss: float, test_loss: float, is_best: bool = False):
        """Save training checkpoint"""
        checkpoint = {
            'epoch': epoch,
            'train_loss': train_loss,
            'test_loss': test_loss,
            'temperature': self.temperature.data.item(),
            'training_stats': self.training_stats
        }

        checkpoint_path = os.path.join(save_path, f"checkpoint_epoch_{epoch}.pt")
        torch.save(checkpoint, checkpoint_path)

        if is_best:
            best_path = os.path.join(save_path, "best_checkpoint.pt")
            torch.save(checkpoint, best_path)
            print(f"Saved best checkpoint at epoch {epoch}")

    def fine_tune(self, csv_path: str, image_folder: str, save_path: str = "finetuned_lora_clip"):
        """Complete fine-tuning pipeline with LoRA"""
        os.makedirs(save_path, exist_ok=True)

        config = TrainingConfig()
        print(f"Training configuration:")
        print(f"  - Batch Size: {config.batch_size}")
        print(f"  - Learning Rate: {config.learning_rate:.2e}")
        print(f"  - Accumulation Steps: {config.accumulation_steps}")
        print(f"  - LoRA r: {config.lora_r}")
        print(f"  - LoRA alpha: {config.lora_alpha}")

        # Apply LoRA
        self.apply_lora(config)

        # Create dataloaders
        print("Creating dataloaders...")
        train_loader, test_loader = self.create_dataloaders(csv_path, image_folder, config)

        # Setup optimizer - INCLUDE PROJECTION LAYER
        trainable_params = (
            [p for p in self.vision_encoder.parameters() if p.requires_grad] +
            [p for p in self.text_encoder.parameters() if p.requires_grad] +
            list(self.clip_model.vision_projection.parameters()) +
            list(self.clip_model.text_projection.parameters()) +
            [self.temperature]
        )
        optimizer = AdamW(trainable_params, lr=config.learning_rate, weight_decay=config.weight_decay)

        # Learning rate scheduler
        total_steps = len(train_loader) * config.epochs // config.accumulation_steps
        scheduler = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=config.learning_rate/10)

        # Training loop
        best_test_loss = float('inf')
        patience_counter = 0

        for epoch in range(config.epochs):
            print(f"\nEpoch {epoch+1}/{config.epochs}")

            # Train
            train_loss = self.train_epoch(train_loader, optimizer, scheduler, epoch+1, config)

            # Test
            test_loss = self.test_epoch(test_loader)

            # Update training stats
            current_temp = torch.clamp(self.temperature, min=0.01, max=1.0).item()
            current_lr = scheduler.get_last_lr()[0] if hasattr(scheduler, 'get_last_lr') else config.learning_rate

            self.training_stats['train_losses'].append(train_loss)
            self.training_stats['test_losses'].append(test_loss)
            self.training_stats['learning_rates'].append(current_lr)
            self.training_stats['temperatures'].append(current_temp)

            print(f"Results:")
            print(f"  - Train Loss: {train_loss:.4f}")
            print(f"  - Test Loss: {test_loss:.4f}")
            print(f"  - Temperature: {current_temp:.4f}")
            print(f"  - Learning Rate: {current_lr:.2e}")

            # Save checkpoint
            is_best = test_loss < best_test_loss
            self.save_checkpoint(save_path, epoch+1, train_loss, test_loss, is_best)

            # Early stopping
            if is_best:
                best_test_loss = test_loss
                patience_counter = 0
                self.save_model(save_path)
                print(f"New best model! Test loss: {test_loss:.4f}")
            else:
                patience_counter += 1
                print(f"No improvement. Patience: {patience_counter}/{config.patience}")

                if patience_counter >= config.patience:
                    print("Early stopping triggered!")
                    break

        print(f"\nTraining completed!")
        print(f"Best test loss: {best_test_loss:.4f}")

        # Save final training stats
        stats_path = os.path.join(save_path, "training_stats.json")
        with open(stats_path, 'w') as f:
            json.dump(self.training_stats, f, indent=2)

    def save_model(self, save_path: str):
        """FIXED: Enhanced model saving for LoRA - only save what was trained"""
        os.makedirs(save_path, exist_ok=True)

        try:
            # Save LoRA adapters (ONLY TEXT ENCODER HAS LORA)
            if self.lora_applied_to_text_only:
                self.text_encoder.save_pretrained(os.path.join(save_path, "text_lora"))
                print("Saved text LoRA adapters")

            # Save vision encoder state dict (no LoRA)
            torch.save(self.vision_encoder.state_dict(), os.path.join(save_path, "vision_encoder.pt"))
            print("Saved vision encoder state dict")

            # Save tokenizer and processor
            self.tokenizer.save_pretrained(os.path.join(save_path, "tokenizer"))
            self.processor.save_pretrained(os.path.join(save_path, "processor"))

            # Save projection layers
            projection_state = {
                'vision_projection': self.clip_model.vision_projection.state_dict(),
                'text_projection': self.clip_model.text_projection.state_dict(),
            }
            torch.save(projection_state, os.path.join(save_path, "projection_layers.pt"))

            # Save additional parameters
            torch.save({
                'temperature': self.temperature.data.item(),
                'training_stats': self.training_stats,
                'base_model_name': self.model_name,
                'lora_applied_to_text_only': self.lora_applied_to_text_only
            }, os.path.join(save_path, "additional_params.pt"))

            # Save model configuration
            config_data = {
                'model_type': 'LoRA_CLIP',
                'base_model': self.model_name,
                'temperature': self.temperature.data.item(),
                'device': str(self.device),
                'mixed_precision': self.use_mixed_precision,
                'lora_applied_to_text_only': self.lora_applied_to_text_only
            }

            with open(os.path.join(save_path, "config.json"), 'w') as f:
                json.dump(config_data, f, indent=2)

            print(f"LoRA CLIP model saved to {save_path}")

        except Exception as e:
            print(f"Failed to save model: {e}")
            raise

    def load_model(self, model_path: str):
        """FIXED: Enhanced model loading for LoRA - only load what was saved"""
        try:
            # Load additional parameters first to get base model name
            checkpoint = torch.load(os.path.join(model_path, "additional_params.pt"), map_location=self.device)
            base_model_name = checkpoint.get('base_model_name', self.model_name)
            self.lora_applied_to_text_only = checkpoint.get('lora_applied_to_text_only', True)

            # Load base models first
            self.vision_encoder = CLIPVisionModel.from_pretrained(base_model_name).to(self.device)
            self.text_encoder = CLIPTextModel.from_pretrained(base_model_name).to(self.device)

            # Load vision encoder weights (no LoRA applied)
            vision_state_path = os.path.join(model_path, "vision_encoder.pt")
            if os.path.exists(vision_state_path):
                vision_state = torch.load(vision_state_path, map_location=self.device)
                self.vision_encoder.load_state_dict(vision_state)
                print("Loaded vision encoder weights")

            # Load LoRA adapters ONLY for text encoder
            if self.lora_applied_to_text_only:
                text_lora_path = os.path.join(model_path, "text_lora")
                if os.path.exists(text_lora_path):
                    self.text_encoder = PeftModel.from_pretrained(
                        self.text_encoder,
                        text_lora_path
                    )
                    print("Loaded text LoRA adapters")
                else:
                    print("Text LoRA path not found, using base text encoder")

            # Load tokenizer and processor
            self.tokenizer = CLIPTokenizer.from_pretrained(
                os.path.join(model_path, "tokenizer")
            )
            self.processor = CLIPProcessor.from_pretrained(
                os.path.join(model_path, "processor")
            )

            # Create wrapper model
            self.clip_model = CLIPWithLoRAFixed(self.vision_encoder, self.text_encoder)

            # Load projection layers
            projection_path = os.path.join(model_path, "projection_layers.pt")
            if os.path.exists(projection_path):
                projection_state = torch.load(projection_path, map_location=self.device)
                self.clip_model.vision_projection.load_state_dict(projection_state['vision_projection'])
                self.clip_model.text_projection.load_state_dict(projection_state['text_projection'])
                print("Loaded projection layers")

            # Create new temperature parameter
            self.temperature = nn.Parameter(
                torch.tensor(checkpoint['temperature'], device=self.device, requires_grad=True)
            )

            if 'training_stats' in checkpoint:
                self.training_stats = checkpoint['training_stats']

            print(f"LoRA CLIP model loaded from {model_path}")

        except Exception as e:
            print(f"Failed to load model: {e}")
            raise


In [17]:
class PersianMultimodalEmbeddingExtractor:
    """Extract embeddings from fine-tuned LoRA CLIP model"""

    def __init__(self, model_path: str, device: str = None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model_path = model_path

        # Load the fine-tuned LoRA model
        self.fine_tuner = LoRACLIPFineTuner(device=self.device, use_mixed_precision=False)
        self.fine_tuner.load_model(model_path)

        # Set to evaluation mode
        self.fine_tuner.vision_encoder.eval()
        self.fine_tuner.text_encoder.eval()

        print(f"Embedding extractor ready on {self.device}")

    def extract_text_embeddings(self, texts: List[str], batch_size: int = 32) -> np.ndarray:
        """Extract embeddings from text"""
        all_embeddings = []

        with torch.no_grad():
            for i in range(0, len(texts), batch_size):
                batch_texts = texts[i:i+batch_size]

                # Tokenize batch
                inputs = self.fine_tuner.tokenizer(
                    batch_texts,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=77  # CLIP max length
                ).to(self.device)

                # Get embeddings using the wrapper
                _, embeddings = self.fine_tuner.clip_model.encode_text(
                    inputs['input_ids'],
                    inputs['attention_mask']
                )

                # Normalize embeddings
                embeddings = F.normalize(embeddings, dim=1)

                all_embeddings.append(embeddings.cpu().numpy())

        return np.vstack(all_embeddings)

    def extract_image_embeddings(self, image_paths: List[str], batch_size: int = 32) -> np.ndarray:
        """Extract embeddings from images"""
        all_embeddings = []

        with torch.no_grad():
            for i in range(0, len(image_paths), batch_size):
                batch_paths = image_paths[i:i+batch_size]
                batch_images = []

                # Load and preprocess images
                for path in batch_paths:
                    try:
                        image = Image.open(path).convert('RGB')
                        batch_images.append(image)
                    except Exception as e:
                        print(f"Failed to load {path}: {e}")
                        # Use dummy image
                        batch_images.append(Image.new('RGB', (224, 224), color='white'))

                # Preprocess batch
                inputs = self.fine_tuner.processor(
                    images=batch_images,
                    return_tensors="pt"
                ).to(self.device)

                # Get embeddings using the wrapper
                _, embeddings = self.fine_tuner.clip_model.encode_image(inputs['pixel_values'])

                # Normalize embeddings
                embeddings = F.normalize(embeddings, dim=1)

                all_embeddings.append(embeddings.cpu().numpy())

        return np.vstack(all_embeddings)

    def compute_similarity(self, text_embeddings: np.ndarray, image_embeddings: np.ndarray) -> np.ndarray:
        """Compute cosine similarity between text and image embeddings"""
        # Ensure embeddings are normalized
        text_embeddings = text_embeddings / np.linalg.norm(text_embeddings, axis=1, keepdims=True)
        image_embeddings = image_embeddings / np.linalg.norm(image_embeddings, axis=1, keepdims=True)

        return np.dot(text_embeddings, image_embeddings.T)


In [18]:
class PersianVectorDatabase:
    """Vector database for Persian celebrity embeddings"""

    def __init__(self, db_type: str = "faiss", db_path: str = "persian_vector_db"):
        self.db_type = db_type
        self.db_path = db_path
        self.embedding_dim = 512  # CLIP embedding dimension

        if db_type == "faiss":
            self._init_faiss()
        elif db_type == "chromadb":
            self._init_chromadb()
        else:
            raise ValueError(f"Unsupported database type: {db_type}")

        self.metadata = []  # Store celebrity info
        self.is_built = False

    def _init_faiss(self):
        """Initialize FAISS index"""
        self.index = faiss.IndexFlatIP(self.embedding_dim)  # Inner product for cosine similarity

    def _init_chromadb(self):
        """Initialize ChromaDB"""
        try:
            import chromadb
        except ImportError:
            raise ImportError("ChromaDB not installed. Install with pip install chromadb")
        self.client = chromadb.PersistentClient(path=self.db_path)
        self.collection = self.client.get_or_create_collection(
            name="persian_celebrities",
            metadata={"hnsw:space": "cosine"}
        )

    def add_embeddings(self, embeddings: np.ndarray, metadata: List[Dict]):
        """Add embeddings to database"""
        if self.db_type == "faiss":
            # Normalize for cosine similarity
            normalized_embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
            self.index.add(normalized_embeddings.astype(np.float32))
            self.metadata.extend(metadata)

        elif self.db_type == "chromadb":
            # ChromaDB expects lists
            embeddings_list = embeddings.tolist()
            ids = [f"celeb_{i}" for i in range(len(embeddings))]
            documents = [meta.get('description', '') for meta in metadata]
            metadatas = metadata

            self.collection.add(
                embeddings=embeddings_list,
                ids=ids,
                documents=documents,
                metadatas=metadatas
            )

        self.is_built = True
        print(f"Added {len(embeddings)} embeddings to {self.db_type} database")

    def search(self, query_embedding: np.ndarray, top_k: int = 5) -> List[Dict]:
        """Search for similar embeddings"""
        if not self.is_built:
            raise RuntimeError("Database not built yet. Add embeddings first.")

        query_embedding = query_embedding.reshape(1, -1)

        if self.db_type == "faiss":
            # Normalize query
            query_embedding = query_embedding / np.linalg.norm(query_embedding)
            scores, indices = self.index.search(query_embedding.astype(np.float32), top_k)

            results = []
            for score, idx in zip(scores[0], indices[0]):
                if idx < len(self.metadata):
                    result = self.metadata[idx].copy()
                    result['similarity'] = float(score)
                    results.append(result)

            return results

        elif self.db_type == "chromadb":
            results = self.collection.query(
                query_embeddings=query_embedding.tolist(),
                n_results=top_k
            )

            formatted_results = []
            for i in range(len(results['ids'][0])):
                result = {
                    'id': results['ids'][0][i],
                    'similarity': 1 - results['distances'][0][i],  # Convert distance to similarity
                    'metadata': results['metadatas'][0][i],
                    'document': results['documents'][0][i]
                }
                formatted_results.append(result)

            return formatted_results

    def save(self, path: str = None):
        """Save database to disk"""
        save_path = path or self.db_path
        os.makedirs(save_path, exist_ok=True)

        if self.db_type == "faiss":
            faiss.write_index(self.index, os.path.join(save_path, "index.faiss"))
            with open(os.path.join(save_path, "metadata.pkl"), 'wb') as f:
                pickle.dump(self.metadata, f)
        # ChromaDB is already persistent

        print(f"Database saved to {save_path}")

    def load(self, path: str = None):
        """Load database from disk"""
        load_path = path or self.db_path

        if self.db_type == "faiss":
            index_path = os.path.join(load_path, "index.faiss")
            metadata_path = os.path.join(load_path, "metadata.pkl")
            if os.path.exists(index_path) and os.path.exists(metadata_path):
                self.index = faiss.read_index(index_path)
                with open(metadata_path, 'rb') as f:
                    self.metadata = pickle.load(f)
                self.is_built = True
                print(f"FAISS database loaded from {load_path}")
            else:
                print(f"No saved FAISS index or metadata found at {load_path}")
        # ChromaDB loads automatically from persistent path


In [19]:
class Gemma3VisionLanguageModel:
    """Fixed Gemma 3 Vision-Language Model for Persian Celebrity RAG"""

    def __init__(self, model_name: str = "google/gemma-2-2b-it", device: str = None):
        self.model_name = model_name
        if model_name == "gemini-2.5-flash":
            self.client = genai.Client()
        else:
            torch._dynamo.config.disable = True
            torch.backends.cuda.enable_math_sdp(False)
            self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")

            print(f"Initializing Gemma 3 model: {model_name}")
            self._load_model()

    def _load_model(self):
        """Load Gemma 3 vision-language model"""
        try:
            # For now, using Gemma 2 as Gemma 3 multimodal may not be widely available yet
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                torch_dtype=torch.float16,
                device_map="auto",
                trust_remote_code=True,
                attn_implementation="eager"

            )

            self.tokenizer = AutoTokenizer.from_pretrained(
                self.model_name,
                trust_remote_code=True
            )

            # Ensure pad token is set
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token

            print(f"Successfully loaded Gemma model: {self.model_name}")

        except Exception as e:
            print(f"Failed to load Gemma model: {e}")
            # Fallback to a simpler model
            self._load_fallback_model()

    def _load_fallback_model(self):
        """Load fallback Gemma model without multimodal features"""
        try:
            fallback_model = "google/gemma-2-2b-it"
            print(f"Loading fallback model: {fallback_model}")

            self.model = AutoModelForCausalLM.from_pretrained(
                fallback_model,
                torch_dtype=torch.bfloat16,
                device_map="auto",
                trust_remote_code=True
            )

            self.tokenizer = AutoTokenizer.from_pretrained(fallback_model)

            # Ensure pad token is set
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token

            self.model_name = fallback_model

            print(f"Successfully loaded fallback model: {fallback_model}")

        except Exception as e:
            print(f"Failed to load fallback model: {e}")
            raise RuntimeError("Unable to load any Gemma model")

    def generate_response(self, prompt: str, image: Image.Image = None, **kwargs) -> str:
        """Generate response using Gemma model - FIXED VERSION"""
        try:
            # Tokenize the prompt
            inputs = self.tokenizer(
                prompt,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=2048
            ).to(self.device)

            input_ids = inputs.input_ids
            attention_mask = inputs.attention_mask

            # Get the length of input for proper slicing
            input_length = input_ids.shape[1]

            # Generate response
            with torch.no_grad():
                outputs = self.model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=kwargs.get('max_tokens', 500),
                    temperature=kwargs.get('temperature', 0.7),
                    do_sample=True,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                    repetition_penalty=1.1,
                    top_p=0.9
                )

            # Decode only the new tokens (response part)
            # Fix: Properly handle tensor slicing
            if len(outputs.shape) == 2 and outputs.shape[0] == 1:
                # Single batch case
                response_tokens = outputs[0, input_length:]
            else:
                # Multi-batch case or different tensor structure
                response_tokens = outputs[:, input_length:]
                if response_tokens.dim() == 2:
                    response_tokens = response_tokens[0]

            # Decode the response
            response = self.tokenizer.decode(response_tokens, skip_special_tokens=True)

            # Clean up the response
            response = response.strip()

            return response

        except Exception as e:
            print(f"Failed to generate response: {e}")
            import traceback
            print(f"Traceback: {traceback.format_exc()}")
            return "متأسفم، نمی‌توانم در این لحظه پاسخ مناسبی تولید کنم."

    def generate_persian_response(self, context: str, query: str, **kwargs) -> str:
        """Generate Persian response with better prompting"""

        system_prompt = """شما یک دستیار هوشمند هستید که درباره چهره‌های مشهور ایرانی اطلاعات می‌دهد."""

        user_prompt = f""".بر اساس دانش خود و اطلاعات زیر، به سوال پاسخ دهید:

        فقط در کلمه یکی از گزینه ها را بیان کنید یعنی یک عدد از 1 تا 4.

        اطلاعات موجود:
        {context}

        سوال: {query}

        پاسخ:"""

        full_prompt = f"{system_prompt}\n\n{user_prompt}"

        if self.model_name == "gemini-2.5-flash":
            config = types.GenerateContentConfig(
                temperature=kwargs.get('temperature', 0.7),
                top_p=0.95,
                max_output_tokens=kwargs.get('max_tokens', 400),
                # If you want to adjust “thinking” behavior:
                thinking_config=types.ThinkingConfig(thinking_budget=0)  # disable thinking
            )

            # Generate content
            response = self.client.models.generate_content(
                model="gemini-2.5-flash",
                contents=full_prompt,
                config=config
            )

            return response.text
        else:
            return self.generate_response(
                prompt=full_prompt,
                max_tokens=kwargs.get('max_tokens', 400),
                temperature=kwargs.get('temperature', 0.7)
            )


In [20]:
class PersianCelebrityRAGSystem:
    """Complete RAG system with LoRA CLIP retrieval and Gemma 3 generation"""

    def __init__(self, clip_model_path: str, vector_db_path: str,
                 gemma_model_name: str = "google/gemma-2-2b-it"):

        self.clip_model_path = clip_model_path
        self.vector_db_path = vector_db_path

        self.gemma_model_name = gemma_model_name

        # Initialize components
        self._initialize_system()

    def _initialize_system(self):
        """Initialize all system components"""
        try:
            # Load CLIP retrieval components
            print("Loading LoRA CLIP embedding extractor...")
            self.embedding_extractor = PersianMultimodalEmbeddingExtractor(self.clip_model_path)

            # Load vector database
            print("Loading vector database...")
            self.vector_db = PersianVectorDatabase(db_type="faiss", db_path=self.vector_db_path)
            self.vector_db.load()

            # Load Gemma 3 model
            print("Loading vision-language model...")
            self.gemma_model = Gemma3VisionLanguageModel(self.gemma_model_name)

            print("Persian Celebrity RAG System with LoRA CLIP initialized successfully!")

        except Exception as e:
            print(f"Failed to initialize system: {e}")
            raise

    def search_celebrities(self, query: str, top_k: int = 5) -> List[Dict]:
        """Search for celebrities using text query"""
        # Extract query embedding
        query_embedding = self.embedding_extractor.extract_text_embeddings([query])

        # Search database
        results = self.vector_db.search(query_embedding[0], top_k=top_k)

        return results

    def search_celebrities_multimodal(self, image_path: str, top_k: int = 5) -> List[Dict]:
        """Search for celebrities using text query"""
        # Extract query embedding
        query_embedding = self.embedding_extractor.extract_image_embeddings([image_path])

        # Search database
        results = self.vector_db.search(query_embedding[0], top_k=top_k)

        return results

    def generate_celebrity_response(self, query: str, search_results: List[Dict],
                                  context_image: Image.Image = None) -> str:
        """Generate response about celebrities using Gemma 3"""

        # Create context from search results
        context_parts = []
        for i, result in enumerate(search_results[:3], 1):  # Use top 3 results
            if 'metadata' in result:
                metadata = result['metadata']
            else:
                metadata = {k: v for k, v in result.items() if k != 'similarity'}

            name = metadata.get('name', 'نامشخص')
            description = metadata.get('description', metadata.get('question', ''))
            similarity = result.get('similarity', 0)

            context_parts.append(f"{i}. {name} (مشابهت: {similarity:.3f})\n   توضیحات: {description}")

        context = "\n\n".join(context_parts)
        # can here add image context too!!!

        # Use the improved Persian response generation
        response = self.gemma_model.generate_persian_response(
            context=context,
            query=query,
            max_tokens=400,
            temperature=0.7
        )

        return response

    def generate_celebrity_response_multimodal(self, query: str, search_results: List[Dict],
                                  context_image_path: str) -> str:
        """Generate response about celebrities using Gemma 3"""

        # Create context from search results
        context_parts = []
        for i, result in enumerate(search_results[:3], 1):  # Use top 3 results
            if 'metadata' in result:
                metadata = result['metadata']
            else:
                metadata = {k: v for k, v in result.items() if k != 'similarity'}

            name = metadata.get('name', 'نامشخص')
            description = metadata.get('description', metadata.get('question', ''))
            similarity = result.get('similarity', 0)

            context_parts.append(f"{i}. {name} (مشابهت: {similarity:.3f})\n   توضیحات: {description}")

        context = "\n\n".join(context_parts)

        system_prompt = """شما یک دستیار هوشمند هستید که درباره چهره‌های مشهور ایرانی اطلاعات می‌دهد."""

        user_prompt = f""".بر اساس دانش خود و اطلاعات زیر، به سوال پاسخ دهید:

        فقط در کلمه یکی از گزینه ها را بیان کنید یعنی یک عدد از 1 تا 4.

        این سوال شامل یک تصویر است.

        اطلاعات موجود:
        {context}

        سوال: {query}

        پاسخ:"""

        full_prompt = f"{system_prompt}\n\n{user_prompt}"

        with open(context_image_path, "rb") as f:
            image_bytes = f.read()

        image_part = types.Part.from_bytes(
            data=image_bytes,
            mime_type="image/jpeg"   # use correct MIME type
        )
        contents = [
            image_part,
            full_prompt
        ]

        config = types.GenerateContentConfig(
            temperature=0.7,
            top_p=0.95,
            max_output_tokens=400,
            # If you want to adjust “thinking” behavior:
            thinking_config=types.ThinkingConfig(thinking_budget=0)  # disable thinking
        )

        # Generate content
        response = baseline_client.models.generate_content(
            model="gemini-2.5-flash",
            contents=contents,
            config=config
        ).text

        return response

    def process_query(self, query: str, image: Image.Image = None, top_k: int = 5) -> Dict:
        """Complete RAG pipeline: search + generate"""

        # Search for relevant celebrities
        search_results = self.search_celebrities(query, top_k=top_k)

        # Generate response
        response = self.generate_celebrity_response(query, search_results, image)

        return {
            'query': query,
            'search_results': search_results,
            'response': response,
            'total_results': len(search_results)
        }

    def process_multimodal_query(self, query: str, image_path: str = None, top_k: int = 5) -> Dict:
        """Complete RAG pipeline: search + generate"""

        # Search for relevant celebrities
        search_results = self.search_celebrities_multimodal(image_path, top_k=top_k)

        # Generate response
        response = self.generate_celebrity_response_multimodal(query, search_results, image_path)

        return {
            'query': query,
            'search_results': search_results,
            'response': response,
            'total_results': len(search_results)
        }


In [21]:
def _assert_path(p):
    if not os.path.exists(p):
        raise FileNotFoundError(f"Path not found: {p}")

def _faiss_index_exists(p):
    return os.path.exists(os.path.join(p, "index.faiss"))

def create_demo_system(clip_model_path: str, vector_db_path: str, model_name: str = "google/gemma-2-2b-it") -> PersianCelebrityRAGSystem:
    """Create a demo RAG system with LoRA CLIP"""
    return PersianCelebrityRAGSystem(
        clip_model_path=clip_model_path,
        vector_db_path=vector_db_path,
        gemma_model_name= model_name
    )


### tested models:

- gen_model_name = "google/gemma-3-4b-it"
- gen_model_name = "google/gemma-3-1b-it"

- gen_model_name = "Qwen/Qwen2.5-VL-7B-Instruct"
- gen_model_name = "Qwen/Qwen2.5-VL-3B-Instruct"

**but the results was not good so we use the "gemini-2.5-flash"**

In [ ]:
# # tested models
# gen_model_name = "google/gemma-3-4b-it"
# gen_model_name = "google/gemma-3-1b-it"

# gen_model_name = "Qwen/Qwen2.5-VL-7B-Instruct"
# gen_model_name = "Qwen/Qwen2.5-VL-3B-Instruct"
# but the results was not good so we use the "gemini-2.5-flash"

### testing main pipeline works properly

In [ ]:
DB_TYPE = "faiss"
MODEL_SAVE_PATH = "/content/drive/MyDrive/NLP_HW3_phase2/FineTuned_LoRA_CLIP"
DB_PATH =  "/content/drive/MyDrive/NLP_HW3_phase2/Persian_Vector_DB"

if not os.path.exists(os.path.join(MODEL_SAVE_PATH, "config.json")):
    print("No trained model found")

    # Initialize fine-tuner
    fine_tuner = LoRACLIPFineTuner(
        model_name="openai/clip-vit-base-patch32",
        device="cuda" if torch.cuda.is_available() else "cpu"
    )

    # Fine-tune the model
    fine_tuner.fine_tune(
        csv_path=CSV_PATH,
        image_folder=IMAGE_FOLDER,
        save_path=MODEL_SAVE_PATH
    )

    print("LoRA CLIP fine-tuning completed!")
else:
    print("Existing LoRA CLIP model found — will be loaded by the RAG system.")

if DB_TYPE == "faiss" and not _faiss_index_exists(DB_PATH):
    print("No FAISS index found")

    # Uses the fine-tuned LoRA CLIP saved in MODEL_SAVE_PATH
    vector_db, extractor = build_persian_celebrity_database(
        csv_path=CSV_PATH,
        image_folder=IMAGE_FOLDER,
        model_path=MODEL_SAVE_PATH,
        db_type=DB_TYPE,
        db_path=DB_PATH
    )

    print("Vector DB built and saved.")
else:
    print("Existing vector DB detected — will be loaded by the RAG system.")

print("Creating Persian Celebrity RAG System with LoRA CLIP...")
# rag_system = create_demo_system(MODEL_SAVE_PATH, DB_PATH, "google/gemma-3-1b-it")
# rag_system = create_demo_system(MODEL_SAVE_PATH, DB_PATH, "google/gemma-2-2b-it")
# rag_system = create_demo_system(MODEL_SAVE_PATH, DB_PATH, "google/gemma-3-4b-it")
# rag_system = create_demo_system(MODEL_SAVE_PATH, DB_PATH, "Qwen/Qwen2.5-VL-3B-Instruct")
# rag_system = create_demo_system(MODEL_SAVE_PATH, DB_PATH, "Qwen/Qwen2.5-VL-7B-Instruct")
rag_system = create_demo_system(MODEL_SAVE_PATH, DB_PATH, "gemini-2.5-flash")

# --- quick single query test ---
query = "حسام‌الدین آشنا که بود؟"
result = rag_system.process_query(query)
print(result["response"])

# --- batch tests ---
print("Testing RAG system with multiple queries...")
test_queries = [
    """
    محمد معتضد باهری در چه سالی متولد شد؟
    1- ۱۲۹۰ خورشیدی
    2- ۱۲۹۷ خورشیدی
    3- ۱۳۰۲ خورشیدی
    4- ۱۳۸۶ خورشیدی
    """,
]

for q in test_queries:
    out = rag_system.process_query(q)

    print(f"\nسوال: {q}")
    print(f"پاسخ: {out['response']}")
    print("-" * 60)

Existing LoRA CLIP model found — will be loaded by the RAG system.
Existing vector DB detected — will be loaded by the RAG system.
Creating Persian Celebrity RAG System with LoRA CLIP...
Loading LoRA CLIP embedding extractor...
Loading CLIP model: openai/clip-vit-base-patch32


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

... (3 outputs left out)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

Loaded CLIP models on cuda
Loaded vision encoder weights
Loaded text LoRA adapters
Initialized CLIPWithLoRAFixed: vision_dim=768, text_dim=512, common_dim=512
Loaded projection layers
LoRA CLIP model loaded from /content/drive/MyDrive/NLP_HW3_phase2/FineTuned_LoRA_CLIP
Embedding extractor ready on cuda
Loading vector database...
FAISS database loaded from /content/drive/MyDrive/NLP_HW3_phase2/Persian_Vector_DB
Loading vision-language model...
Persian Celebrity RAG System with LoRA CLIP initialized successfully!
4
Testing RAG system with multiple queries...

سوال: 
    محمد معتضد باهری در چه سالی متولد شد؟
    1- ۱۲۹۰ خورشیدی
    2- ۱۲۹۷ خورشیدی
    3- ۱۳۰۲ خورشیدی
    4- ۱۳۸۶ خورشیدی
    
پاسخ: 2
------------------------------------------------------------


### Load test data

In [23]:
with open("/content/drive/MyDrive/NLP_HW3_phase3/Question_only_Text.json", "r") as f:
    text_questions = json.load(f)

with open("/content/drive/MyDrive/NLP_HW3_phase3/Question_Text_Image.json", "r") as f:
    multimodal_questions = json.load(f)

### Utility functions

In [24]:
def extract_answer_choice(text):
    """Extract answer choice (1-4) from model output."""
    if text is None:
        return None
    match = re.search(r'[1-4]', text)
    return int(match.group()) if match else None

def compute_accuracy(questions, predictions):
    """Compute overall accuracy, category-wise accuracy, and errors."""
    correct = 0
    per_category = {}

    for i, q in enumerate(questions):
        category = q["subject_category"]
        if category not in per_category:
            per_category[category] = {"total": 0, "correct": 0}

        per_category[category]["total"] += 1
        if predictions[i] == q["answer"]:
            correct += 1
            per_category[category]["correct"] += 1

    overall_accuracy = correct / len(questions)
    category_accuracy = {cat: counts["correct"] / counts["total"] for cat, counts in per_category.items()}
    micro_error = 1 - overall_accuracy
    macro_error = 1 - sum(category_accuracy.values()) / len(category_accuracy)

    return overall_accuracy, category_accuracy, micro_error, macro_error

### Evaluation functions using multimodal RAG system

In [25]:
def evaluate_with_rag_system(questions, is_multimodal=False):
    """Evaluate using your RAG system's process_query method."""
    predictions = []

    desc = "Evaluating " + ("multimodal" if is_multimodal else "text") + " questions"

    all_retrivied = []

    for q in tqdm(questions, desc=desc, unit="question"):
        question = q["question"]
        iteration = q["iteration"]

        query = f"""
            لطفا فقط یکی از گزینه‌های 1، 2، 3 یا 4 را انتخاب کنید و تنها عدد مربوطه را ارائه دهید.
            سوال: {question[0]}
            {question[1]}
            {question[2]}
            {question[3]}
            {question[4]}
            """

        if is_multimodal:
            image_path = f"/content/drive/MyDrive/NLP_HW3_phase3/Image/{iteration}.jpeg"
            result = rag_system.process_multimodal_query(query, image_path)

        else:
            # Process the query with your RAG system
            result = rag_system.process_query(query)

        response = result["response"]
        total_results = result["total_results"]
        search_results = result["search_results"]
        all_retrivied.append(search_results)

        # Extract answer choice
        pred_choice = extract_answer_choice(response)
        predictions.append(pred_choice)

        time.sleep(7)

    overall_accuracy, category_accuracy, micro_error, macro_error = compute_accuracy(questions, predictions)

    return overall_accuracy, category_accuracy, micro_error, macro_error, all_retrivied

In [26]:
baseline_client = genai.Client()

def evaluate_baseline(questions, is_multimodal=False):
    """Evaluate baseline model without RAG."""
    predictions = []

    # Add progress bar
    for q in tqdm(questions, desc="Evaluating Baseline", unit="question"):
        iteration = q["iteration"]

        question = q["question"]

        query = f"""
                لطفا فقط یکی از گزینه‌های 1، 2، 3 یا 4 را انتخاب کنید و تنها عدد مربوطه را ارائه دهید.
                سوال: {question[0]}
                {question[1]}
                {question[2]}
                {question[3]}
                {question[4]}
            """

        system_prompt = """شما یک دستیار هوشمند هستید که درباره چهره‌های مشهور ایرانی اطلاعات می‌دهد."""

        user_prompt = f"""فقط در کلمه یکی از گزینه ها را بیان کنید یعنی یک عدد از 1 تا 4.

        {'این سوال شامل یک تصویر است. ' if is_multimodal else ''}

        سوال: {query}

        پاسخ:"""

        full_prompt = f"{system_prompt}\n\n{user_prompt}"

        if is_multimodal:
            image_path = f"/content/drive/MyDrive/NLP_HW3_phase3/Image/{iteration}.jpeg"
            with open(image_path, "rb") as f:
                image_bytes = f.read()

            image_part = types.Part.from_bytes(
                data=image_bytes,
                mime_type="image/jpeg"   # use correct MIME type
            )
            contents = [
                image_part,
                question
            ]
        else:
            contents = [
                question
            ]

        config = types.GenerateContentConfig(
            temperature=0.7,
            top_p=0.95,
            max_output_tokens=400,
            # If you want to adjust “thinking” behavior:
            thinking_config=types.ThinkingConfig(thinking_budget=0)  # disable thinking
        )

        # Generate content
        answer = baseline_client.models.generate_content(
            model="gemini-2.5-flash",
            contents=contents,
            config=config
        ).text

        # Generate answer without context
        pred_choice = extract_answer_choice(answer)
        predictions.append(pred_choice)

        time.sleep(7)

    return compute_accuracy(questions, predictions)

### Run Evaluations

In [27]:
print("Evaluating baseline on text questions...")
text_acc, text_cat_acc, text_micro_err, text_macro_err = evaluate_baseline(text_questions)

print("Evaluating baseline on multimodal questions...")
multi_acc, multi_cat_acc, multi_micro_err, multi_macro_err = evaluate_baseline(multimodal_questions, True)

Evaluating baseline on text questions...


Evaluating Baseline: 100%|██████████| 60/60 [07:55<00:00,  7.93s/question]


Evaluating baseline on multimodal questions...


Evaluating Baseline: 100%|██████████| 60/60 [09:37<00:00,  9.63s/question]


In [28]:
print("\n=== RESULTS ===")
print(f"Baseline Text Questions Accuracy: {text_acc:.4f}")
print(f"Baseline Text Questions Micro Error: {text_micro_err:.4f}")
print(f"Baseline Text Questions Macro Error: {text_macro_err:.4f}")
print("Baseline Text Questions Category Accuracy:", {k: f"{v:.4f}" for k, v in text_cat_acc.items()})

print(f"\nBaseline Multimodal Questions Accuracy: {multi_acc:.4f}")
print(f"Baseline Multimodal Questions Micro Error: {multi_micro_err:.4f}")
print(f"Baseline Multimodal Questions Macro Error: {multi_macro_err:.4f}")
print("Baseline Multimodal Questions Category Accuracy:", {k: f"{v:.4f}" for k, v in multi_cat_acc.items()})


=== RESULTS ===
Baseline Text Questions Accuracy: 0.3333
Baseline Text Questions Micro Error: 0.6667
Baseline Text Questions Macro Error: 0.7469
Baseline Text Questions Category Accuracy: {'سیاسی': '0.3125', 'ورزشی': '0.5000', 'ادبیات': '0.0000', 'هنر': '0.2000'}

Baseline Multimodal Questions Accuracy: 0.2833
Baseline Multimodal Questions Micro Error: 0.7167
Baseline Multimodal Questions Macro Error: 0.5434
Baseline Multimodal Questions Category Accuracy: {'سیاسی': '0.1765', 'ادبیات': '1.0000', 'مذهبی': '0.5000', 'هنر': '0.3684', 'ورزشی': '0.2381'}


In [29]:
print("Evaluating RAG on text questions...")
text_rag_acc, text_rag_cat_acc, text_rag_micro_err, text_rag_macro_err, text_retrieved = evaluate_with_rag_system(text_questions, False)

print("Evaluating RAG on multimodal questions...")
multi_rag_acc, multi_rag_cat_acc, multi_rag_micro_err, multi_rag_macro_err, multi_retrieved = evaluate_with_rag_system(multimodal_questions, True)

Evaluating RAG on text questions...


Evaluating text questions: 100%|██████████| 60/60 [07:55<00:00,  7.93s/question]


Evaluating RAG on multimodal questions...


Evaluating multimodal questions: 100%|██████████| 60/60 [08:25<00:00,  8.42s/question]


In [34]:
print("\n=== RESULTS ===")

print(f"\nRAG Text Questions Accuracy: {text_rag_acc:.4f}")
print(f"RAG Text Questions Micro Error: {text_rag_micro_err:.4f}")
print(f"RAG Text Questions Macro Error: {text_rag_macro_err:.4f}")
print("RAG Text Questions Category Accuracy:", {k: f"{v:.4f}" for k, v in text_rag_cat_acc.items()})

print(f"\nRAG Multimodal Questions Accuracy: {multi_rag_acc:.4f}")
print(f"RAG Multimodal Questions Micro Error: {multi_rag_micro_err:.4f}")
print(f"RAG Multimodal Questions Macro Error: {multi_rag_macro_err:.4f}")
print("RAG Multimodal Questions Category Accuracy:", {k: f"{v:.4f}" for k, v in multi_rag_cat_acc.items()})


=== RESULTS ===

RAG Text Questions Accuracy: 0.5167
RAG Text Questions Micro Error: 0.4833
RAG Text Questions Macro Error: 0.5000
RAG Text Questions Category Accuracy: {'سیاسی': '0.4500', 'ورزشی': '0.4500', 'ادبیات': '0.6000', 'هنر': '0.5000'}

RAG Multimodal Questions Accuracy: 0.4667
RAG Multimodal Questions Micro Error: 0.5333
RAG Multimodal Questions Macro Error: 0.5000
RAG Multimodal Questions Category Accuracy: {'سیاسی': '0.4000', 'ادبیات': '0.6000', 'مذهبی': '0.4000', 'هنر': '0.5000', 'ورزشی': '0.6000'}



### Retrieval Metrics

In [50]:
def calculate_retrieval_metrics(retrieved_list, ground_truth_indices):
    """Calculate retrieval metrics (HIT@K, Precision, Recall)."""
    hit_at_k = 0
    precision_sum = 0
    recall_sum = 0

    for retrieved, true_idx in zip(retrieved_list, ground_truth_indices):
        # retrieved_indices = [item["index"] for item in retrieved]

        retrieved_indices = []
        for item in retrieved:
            path = item["image_path"]
            index = path.split("/")[-2]
            retrieved_indices.append(index)

        # Check if true index is in retrieved
        hit_at_k += 1 if true_idx in retrieved_indices else 0

        # Precision and Recall
        relevant_retrieved = 1 if true_idx in retrieved_indices else 0
        precision_sum += relevant_retrieved / len(retrieved_indices)
        recall_sum += relevant_retrieved / 1  # Assuming 1 relevant doc per query

    return {
        "HIT@3": hit_at_k / len(retrieved_list),
        "Precision": precision_sum / len(retrieved_list),
        "Recall": recall_sum / len(retrieved_list)
    }


In [51]:
multi_ground_truth_indices = []
for q in multimodal_questions:
    iteration = q["iteration"]
    index = q["index"]
    multi_ground_truth_indices.append(index)

multi_retrieval_metrics = calculate_retrieval_metrics(multi_retrieved, multi_ground_truth_indices)

In [52]:
print("\nRetrieval Metrics for Multimodal Questions:")
for metric, value in multi_retrieval_metrics.items():
    print(f"{metric}: {value:.4f}")


Retrieval Metrics for Multimodal Questions:
HIT@3: 0.7000
Precision: 0.2333
Recall: 0.7000



In [54]:
text_ground_truth_indices = []
for q in text_questions:
    iteration = q["iteration"]
    index = q["index"]
    ground_truth_indices.append(index)

text_retrieval_metrics = calculate_retrieval_metrics(text_retrieved, text_ground_truth_indices)

In [55]:
print("\nRetrieval Metrics for Only Text Questions:")
for metric, value in text_retrieval_metrics.items():
    print(f"{metric}: {value:.4f}")


Retrieval Metrics for Only Text Questions:
HIT@3: 0.6000
Precision: 0.2000
Recall: 0.6000

